# K-means su GPU — Progetto Advanced Computer Architecture

Notebook per compilare, validare e profilare il progetto su Google Colab.

**Prima di eseguire**: `Runtime → Change runtime type → Hardware accelerator: GPU` (una T4 è sufficiente).

Il notebook esegue nell'ordine:
1. verifica della GPU assegnata e del toolkit CUDA;
2. compilazione delle quattro implementazioni;
3. suite di correttezza (obbligatoria prima di guardare qualunque tempo: misurare un codice sbagliato non ha senso);
4. campagna di benchmark e generazione dei grafici;
5. profiling con Nsight Compute sui due kernel principali.

## 1. Ambiente

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
# Sostituisci con l'URL del tuo repository, oppure carica i file a mano.
REPO = 'https://github.com/santoromarco74/aca_project.git'
BRANCH = 'claude/advanced-architecture-project-fyqlds'

!git clone --branch $BRANCH $REPO aca_project
%cd aca_project
!ls -la

## 2. Compilazione

`-arch` va allineato alla GPU effettivamente assegnata: `sm_75` per la T4, `sm_80` per la A100, `sm_60` per la P100. La cella seguente lo ricava automaticamente dalla compute capability riportata dal driver.

In [ ]:
import subprocess

cc = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader']
).decode().strip().splitlines()[0].replace('.', '')
ARCH = f'sm_{cc}'
print('Architettura target:', ARCH)

SRC = ('src/common.cpp src/kmeans_serial.cpp src/kmeans_omp.cpp '
       'src/kmeans_cuda.cu')
FLAGS = (f'-O3 -std=c++17 -Iinclude -lineinfo -arch={ARCH} -DUSE_CUDA '
         f'-Xcompiler "-O3 -fopenmp"')

!mkdir -p build
!nvcc {FLAGS} {SRC} src/main.cpp -o build/kmeans_bench
!nvcc {FLAGS} {SRC} tests/test_correctness.cpp -o build/kmeans_test
print('Compilazione completata.')

## 3. Correttezza

Verifica che le versioni CUDA producano lo stesso risultato della baseline seriale, e che la baseline stessa ricostruisca la ground truth dei blob sintetici. Deve terminare con `TUTTI I TEST SUPERATI`.

In [ ]:
!./build/kmeans_test

## 4. Benchmark singolo

Prima misura di riferimento: quattro implementazioni sullo stesso dataset e con gli stessi centroidi iniziali.

In [ ]:
!./build/kmeans_bench --n 1000000 --d 32 --k 64 --iter 50 --reps 5

## 5. Campagna completa e grafici

Tre studi di scalabilità (in N, in K, in D) più la scalabilità forte di OpenMP. Richiede qualche minuto.

In [ ]:
!chmod +x scripts/run_benchmarks.sh
!./scripts/run_benchmarks.sh build/kmeans_bench

In [ ]:
!pip install -q pandas matplotlib
!python3 scripts/plot_results.py

In [ ]:
from IPython.display import Image, display
import glob

for path in sorted(glob.glob('results/figures/*.png')):
    print(path)
    display(Image(path))

## 6. Profiling con Nsight Compute

Le metriche da riportare nel report per ciascun kernel:

| Metrica | Cosa dice |
|---|---|
| `sm__throughput.avg.pct_of_peak_sustained_elapsed` | quanto si sfruttano le unità di calcolo |
| `gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed` | quanto si sfrutta la banda verso la DRAM |
| `l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum` | settori letti: **crolla** passando al layout feature-major |
| `sm__warps_active.avg.pct_of_peak_sustained_active` | occupancy raggiunta |

Il confronto fra `assign_naive_kernel` e `assign_opt_kernel` sul numero di settori letti è la prova quantitativa dell'effetto del coalescing: è il grafico/tabella più convincente della sezione.

In [ ]:
# Riepilogo per kernel: tempo, occupancy, throughput di calcolo e memoria.
!ncu --set speedOfLight --kernel-name-base function \
     --launch-count 4 \
     ./build/kmeans_bench --n 1000000 --d 32 --k 64 --iter 10 --reps 0 \
                          --impl cuda_naive,cuda_opt 2>&1 | tail -80

In [ ]:
# Metriche mirate sul traffico di memoria: il confronto diretto naive vs ottimizzato.
!ncu --metrics l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,\
gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed,\
sm__warps_active.avg.pct_of_peak_sustained_active \
     --launch-count 2 \
     ./build/kmeans_bench --n 1000000 --d 32 --k 64 --iter 5 --reps 0 \
                          --impl cuda_naive,cuda_opt 2>&1 | tail -60

## 7. Timeline con Nsight Systems

Serve a mostrare visivamente due cose nel report: che i trasferimenti H2D avvengono **una sola volta** all'inizio, e che la versione naive spende una frazione non trascurabile del tempo nel ping-pong D2H/H2D dei centroidi ad ogni iterazione — costo che la versione ottimizzata azzera finalizzando sul device.

In [ ]:
!nsys profile --stats=true -o results/timeline --force-overwrite true \
     ./build/kmeans_bench --n 1000000 --d 32 --k 64 --iter 20 --reps 0 \
                          --impl cuda_naive,cuda_opt 2>&1 | tail -40

## 8. Validazione incrociata con scikit-learn

Controprova indipendente: l'SSE (inerzia) ottenuto dalla nostra implementazione deve essere allineato a quello di una libreria di riferimento, a parità di dati e numero di cluster.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=200_000, n_features=16, centers=12,
                  cluster_std=1.0, random_state=0)
np.savetxt('results/blobs.csv', X, delimiter=',', fmt='%.6f')

km = KMeans(n_clusters=12, init='k-means++', n_init=1, max_iter=100,
            tol=1e-6, random_state=0).fit(X)
print(f'scikit-learn: inerzia = {km.inertia_:.4f}, iterazioni = {km.n_iter_}')

In [ ]:
!./build/kmeans_bench --dataset results/blobs.csv --k 12 --iter 100 --reps 1

I due valori di inerzia devono coincidere entro pochi decimi percentuali. Una differenza maggiore non indica necessariamente un bug: k-means è sensibile all'inizializzazione, e le due implementazioni estraggono i centroidi iniziali con generatori casuali diversi. Se la differenza è marcata, ripeti con `n_init` più alto su entrambi i lati oppure confronta partendo dagli stessi centroidi.